# 数据读取 （data from Unit03_1_5_select.mat）

## 原始数据的读取

In [11]:
import scipy.io
import numpy as np

# ================= 1️⃣ 读取 .mat 文件 =================
mat_data = scipy.io.loadmat(
    '/home/charles/HZU/Data_processed/multi-condition-transfer-learning/Unit03_1_5_select_3.mat'
)

print("Keys in the .mat file:", mat_data.keys())

# ================= 2️⃣ 取出真实数据 =================
data = mat_data['Unit03_1_5_select_3']

print("Original data shape:", data.shape)

# ================= 3️⃣ 随机采样 n 个样本 =================
n = 5000   # ← 你想采样多少行自己改

num_samples = data.shape[0]

assert n <= num_samples, "n exceeds total samples!"

# 随机不放回采样行索引
idx = np.random.choice(num_samples, size=n, replace=False)

# 采样后的数据
sampled_data = data[idx]

print("Sampled data shape:", sampled_data.shape)

# ================= 4️⃣ 查看部分采样结果 =================
print(sampled_data[:5])




Keys in the .mat file: dict_keys(['__header__', '__version__', '__globals__', 'Unit03_1_5_select_3'])
Original data shape: (14000, 14)
Sampled data shape: (5000, 14)
[[ 6.11716604e+00  1.00884838e+01  2.25669682e-01 -1.32621355e+01
   2.61625458e+02  3.08137451e+02  5.61762268e+02  8.24093140e+02
   8.08289795e+02  8.98853271e+02  7.50059265e+02  7.49740845e+02
   7.49521729e+02  8.33597088e+00]
 [ 6.78343678e+00  1.03760605e+01  2.29583487e-01 -1.04241486e+01
   2.61138580e+02  3.11415985e+02  5.62776306e+02  8.24671143e+02
   7.99115906e+02  8.94981873e+02  7.48362793e+02  7.49371948e+02
   7.48169189e+02  6.87905931e+00]
 [ 6.93191147e+00  9.16153145e+00  1.97594747e-01 -1.17620268e+01
   2.63002319e+02  3.06314056e+02  5.57604004e+02  8.19754761e+02
   8.02170959e+02  8.93877502e+02  7.50068604e+02  7.47768921e+02
   7.48574524e+02  6.74802494e+00]
 [ 6.50737572e+00  9.72194290e+00  2.12904528e-01 -9.99288082e+00
   2.61385284e+02  3.03954865e+02  5.56952148e+02  8.21152893e+02
   

## 特征和标签的分离

In [12]:
import numpy as np

# 假设 'data' 是一个二维数组或矩阵
# 分离特征和标签

# 特征是除了最后一列的数据
X = sampled_data[:, :-1]  # 所有行，去除最后一列

# 标签是最后一列的数据
y = sampled_data[:, -1]  # 所有行，只取最后一列
y = y.reshape(-1, 1)

# # 查看特征和标签
# print("Features (X):")
# print(X[:5])  # 查看前5个特征样本
# print("Labels (y):")
# print(y[:5])  # 查看前5个标签

# 查看特征和标签的形状
print("Shape of Features (X):", X.shape)
print("Shape of Labels (y):", y.shape)

Shape of Features (X): (5000, 13)
Shape of Labels (y): (5000, 1)


## 三集划分

In [13]:
import numpy as np
from sklearn.model_selection import train_test_split

# 假设 X 和 y 是已经分离好的特征和标签
# X: 特征数据，y: 标签数据

# 设置随机种子，确保结果可复现
random_seed = 42

# 控制三集的划分比例：例如 70% 训练集，15% 验证集，15% 测试集
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# 确保划分比例之和为1
assert train_ratio + val_ratio + test_ratio == 1.0, "The sum of ratios must be 1."

# 第一次划分，将训练集和验证+测试集合并
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=1 - train_ratio, random_state=random_seed)

# 第二次划分，将验证集和测试集分开
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=test_ratio / (val_ratio + test_ratio), random_state=random_seed)

# 打印各个数据集的形状
print("Shape of Training Set (X_train, y_train):", X_train.shape, y_train.shape)
print("Shape of Validation Set (X_val, y_val):", X_val.shape, y_val.shape)
print("Shape of Test Set (X_test, y_test):", X_test.shape, y_test.shape)


Shape of Training Set (X_train, y_train): (3499, 13) (3499, 1)
Shape of Validation Set (X_val, y_val): (750, 13) (750, 1)
Shape of Test Set (X_test, y_test): (751, 13) (751, 1)


## 特征按列归一化，

In [14]:
import numpy as np

# ======================= 1️⃣ 计算 train 的均值和标准差 =======================
mu = X_train.mean(axis=0, keepdims=True)     # (1, 13)
std = X_train.std(axis=0, keepdims=True)    # (1, 13)

# 防止某些特征 std=0
std[std == 0] = 1e-8

# ======================= 2️⃣ 三个集合统一归一化 =======================
X_train_norm = (X_train - mu) / std
X_val_norm   = (X_val   - mu) / std
X_test_norm  = (X_test  - mu) / std

print("Normalized shapes:")
print(X_train_norm.shape, X_val_norm.shape, X_test_norm.shape)


Normalized shapes:
(3499, 13) (750, 13) (751, 13)


# Model

## encoder 

In [21]:
import torch
import torch.nn as nn

class PaperEmbedEncoder(nn.Module):
    """
    Paper-style triple embedding encoder (AKGNN embedding part).

    Input:
        X: [B, D]  (B=batch size, D=#variables)
    Output:
        H1, H2, H3: each [B, D, d_embed]
            H1 -> Value for message passing
            H2 -> Query for graph construction
            H3 -> Key   for graph construction
    """
    def __init__(self, D: int, d_embed: int, bias: bool = True):
        super().__init__()
        self.D = D
        self.d_embed = d_embed

        # Three independent linear projections (embed-1/2/3)
        self.proj1 = nn.Linear(D, D * d_embed, bias=bias)  # -> H1
        self.proj2 = nn.Linear(D, D * d_embed, bias=bias)  # -> H2
        self.proj3 = nn.Linear(D, D * d_embed, bias=bias)  # -> H3

    def forward(self, X: torch.Tensor):
        # X: [B, D]
        if X.dim() != 2:
            raise ValueError(f"X must be 2D [B, D], got shape={tuple(X.shape)}")
        if X.size(1) != self.D:
            raise ValueError(f"X second dim must be D={self.D}, got {X.size(1)}")

        B = X.size(0)

        H1 = self.proj1(X).view(B, self.D, self.d_embed)
        H2 = self.proj2(X).view(B, self.D, self.d_embed)
        H3 = self.proj3(X).view(B, self.D, self.d_embed)

        return H1, H2, H3


In [23]:
import torch

# ======================= 1️⃣ 把 numpy 转成 torch =======================
# X_train_norm: (3499, 13)
X_torch = torch.tensor(X_train_norm, dtype=torch.float32)

B, D = X_torch.shape
d_embed = 8   # embedding 维度，你也可以改成 16 / 32

# ======================= 2️⃣ 初始化 encoder =======================
encoder = PaperEmbedEncoder(D=D, d_embed=d_embed)

# ======================= 3️⃣ 前向计算 =======================
with torch.no_grad():
    H1, H2, H3 = encoder(X_torch)

print("H1 shape:", H1.shape)
print("H2 shape:", H2.shape)
print("H3 shape:", H3.shape)
# 期望都是: [3499, 13, d_embed]

# ======================= 4️⃣ 打印前三个“样本”的 embedding =======================

for i in range(3):
    print(f"\n========== Sample {i} ==========")
    print("H1 (Value):")
    print(H1[i])     # shape: [13, d_embed]

    print("\nH2 (Query):")
    print(H2[i])

    print("\nH3 (Key):")
    print(H3[i])


H1 shape: torch.Size([3499, 13, 8])
H2 shape: torch.Size([3499, 13, 8])
H3 shape: torch.Size([3499, 13, 8])

========== Sample 0 ==========
H1 (Value):
tensor([[ 0.5714, -1.1257,  1.3275, -0.0786,  0.1868,  0.0740, -0.1989,  0.4205],
        [-0.4818,  0.0388, -0.5476,  0.2234, -0.1306, -0.2430,  0.6636, -1.0382],
        [-0.9089,  0.2535,  0.2224,  0.0432,  0.7233,  1.0142,  0.5787, -0.1223],
        [ 0.3644,  0.3334,  0.8238,  1.0879,  0.5325,  0.1421, -0.0690, -0.2557],
        [-0.2666, -0.2105,  0.3338, -0.3075, -0.2509,  0.1948, -0.9019,  0.1132],
        [-0.7712,  0.2324, -0.1637, -0.5512,  0.3665, -0.1962,  0.0233, -0.3857],
        [-0.1492,  0.2475,  0.0356,  0.6349,  0.1162,  0.1099,  0.2265, -0.4938],
        [-0.2747,  0.2858, -0.5153,  0.6269,  1.0469,  0.1047, -0.5610, -0.2455],
        [-0.3726, -0.5716,  0.7326, -0.6784,  0.3353, -0.6880,  0.1339, -0.1562],
        [-0.3895, -0.3521, -0.4113,  0.3352,  0.1572,  0.5346,  0.9974, -0.3111],
        [ 0.0392,  0.1675, -